In [1]:
#Setup and Imports
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

C:\Users\LENOVO\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\LENOVO\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Libraries loaded successfully


In [3]:
import os
print(f"Current directory: {os.getcwd()}")

Current directory: d:\sri-lanka-renewable-forecast


In [4]:
# Cell 2: Load Data
import os
from pathlib import Path

# Find the correct path
possible_paths = [
    'data/features_dataset.csv',
    '../data/features_dataset.csv',
    '../../data/features_dataset.csv',
    'features_dataset.csv',
]

df = None
for path in possible_paths:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"✅ Loaded from: {path}")
        break

if df is None:
    print("❌ Could not find features_dataset.csv")
    print(f"Current directory: {os.getcwd()}")
    print("Files in current directory:")
    for f in Path('.').iterdir():
        print(f"  {f.name}")
    raise FileNotFoundError("Please place features_dataset.csv in the data/ folder")

df['Datetime'] = pd.to_datetime(df['Datetime'])
df.set_index('Datetime', inplace=True)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

✅ Loaded from: data/features_dataset.csv
Data shape: (105985, 56)
Date range: 2021-03-30 00:00:00 to 2024-12-31 23:00:00


In [5]:
#Define Renewable Target
# Renewable = Solar + Wind + Major Hydro
df['renewable_target'] = df['solar'] + df['wind'] + df['major hydro']

print(f"Renewable target created")
print(f"Mean: {df['renewable_target'].mean():.1f} MW")
print(f"Std: {df['renewable_target'].std():.1f} MW")
print(f"Min: {df['renewable_target'].min():.1f} MW")
print(f"Max: {df['renewable_target'].max():.1f} MW")

Renewable target created
Mean: 667.2 MW
Std: 293.7 MW
Min: 0.0 MW
Max: 1620.0 MW


In [6]:
#Define Features (Weather + Time Only)
weather_features = ['solar_W_m2', 'temp_C', 'wind_m_s', 'precip_mm', 'humidity_pct']
time_features = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']

# Check if time features exist, if not create them
if 'hour_sin' not in df.columns:
    print("Creating time features...")
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

feature_cols = weather_features + time_features
print(f"Total features: {len(feature_cols)}")
print(feature_cols)

Total features: 12
['solar_W_m2', 'temp_C', 'wind_m_s', 'precip_mm', 'humidity_pct', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']


In [7]:
#Prepare Data (Drop NaN)
X = df[feature_cols].ffill().fillna(0)
y = df['renewable_target']

# Drop rows where target is NaN
clean_mask = ~y.isna()
X = X[clean_mask]
y = y[clean_mask]

print(f"Final dataset: {X.shape[0]} rows, {X.shape[1]} features")
print(f"Target mean: {y.mean():.1f} MW")

Final dataset: 105985 rows, 12 features
Target mean: 667.2 MW


In [8]:
# Cell 4.5: Model Comparison (re-run to confirm)
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd

print("=" * 60)
print("MODEL COMPARISON FOR RENEWABLE FORECAST")
print("=" * 60)

models = {
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, max_depth=7, random_state=42, verbosity=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=7, random_state=42, verbosity=0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Ridge': Ridge(alpha=1.0),
    'Linear Regression': LinearRegression()
}

tscv = TimeSeriesSplit(n_splits=3, test_size=30*96)

results = []
for name, model in models.items():
    fold_mae = []
    print(f"\nTraining {name}...")
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        fold_mae.append(mean_absolute_error(y_test, y_pred))
    
    avg_mae = np.mean(fold_mae)
    results.append({'model': name, 'mae': avg_mae})
    print(f"  Average MAE: {avg_mae:.2f} MW")

# Display results
results_df = pd.DataFrame(results).sort_values('mae')
print("\n" + "=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['model']
best_model_mae = results_df.iloc[0]['mae']
print(f"\n✅ BEST MODEL: {best_model_name} (MAE: {best_model_mae:.2f} MW)")

MODEL COMPARISON FOR RENEWABLE FORECAST

Training LightGBM...
  Average MAE: 246.92 MW

Training XGBoost...
  Average MAE: 254.44 MW

Training Random Forest...
  Average MAE: 253.78 MW

Training Ridge...
  Average MAE: 260.80 MW

Training Linear Regression...
  Average MAE: 260.80 MW

MODEL COMPARISON RESULTS
            model        mae
         LightGBM 246.924650
    Random Forest 253.781655
          XGBoost 254.440858
Linear Regression 260.804470
            Ridge 260.804865

✅ BEST MODEL: LightGBM (MAE: 246.92 MW)


In [ ]:
#Model Comparison for Renewable Forecast
print("=" * 60)
print("MODEL COMPARISON FOR RENEWABLE FORECAST")
print("=" * 60)

# Time series split (3 folds, 30-day test sets)
tscv = TimeSeriesSplit(n_splits=3, test_size=30*96)

models = {
    'LightGBM': lgb.LGBMRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Ridge': Ridge(alpha=1.0),
    'Linear Regression': LinearRegression()
}

comparison_results = []

for name, model in models.items():
    fold_mae = []
    print(f"\nTraining {name}...")
    
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        fold_mae.append(mae)
    
    avg_mae = np.mean(fold_mae)
    comparison_results.append({'model': name, 'mae': avg_mae})
    print(f"  Average MAE: {avg_mae:.2f} MW")

# Display results
results_df = pd.DataFrame(comparison_results).sort_values('mae')
print("\n" + "=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['model']
best_model_mae = results_df.iloc[0]['mae']
print(f"\n✅ BEST MODEL: {best_model_name} (MAE: {best_model_mae:.2f} MW)")

MODEL COMPARISON FOR RENEWABLE FORECAST

Training LightGBM...
  Average MAE: 248.83 MW

Training XGBoost...
  Average MAE: 248.75 MW

Training Random Forest...
  Average MAE: 253.78 MW

Training Ridge...
  Average MAE: 260.80 MW

Training Linear Regression...
  Average MAE: 260.80 MW

MODEL COMPARISON RESULTS
            model        mae
          XGBoost 248.753768
         LightGBM 248.830826
    Random Forest 253.781655
Linear Regression 260.804470
            Ridge 260.804865

✅ BEST MODEL: XGBoost (MAE: 248.75 MW)


In [9]:
#Train Best Model with Cross-Validation
print("=" * 60)
print(f"TRAINING {best_model_name.upper()} - CROSS VALIDATION")
print("=" * 60)

# Select the best model class
if best_model_name == 'LightGBM':
    base_model = lgb.LGBMRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=-1)
elif best_model_name == 'XGBoost':
    base_model = xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=0)
elif best_model_name == 'Random Forest':
    base_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
elif best_model_name == 'Ridge':
    base_model = Ridge(alpha=1.0)
else:
    base_model = LinearRegression()

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=3, test_size=30*96)
cv_results = []
best_model = None
best_fold_mae = float('inf')

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Create fresh model for each fold
    if best_model_name == 'LightGBM':
        model = lgb.LGBMRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=-1)
    elif best_model_name == 'XGBoost':
        model = xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=0)
    elif best_model_name == 'Random Forest':
        model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    elif best_model_name == 'Ridge':
        model = Ridge(alpha=1.0)
    else:
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    cv_results.append(mae)
    
    print(f"Fold {fold+1}: MAE = {mae:.2f} MW")
    
    if mae < best_fold_mae:
        best_fold_mae = mae
        best_model = model

print(f"\nAverage CV MAE: {np.mean(cv_results):.2f} MW")
print(f"Best fold MAE: {best_fold_mae:.2f} MW")

TRAINING LIGHTGBM - CROSS VALIDATION
Fold 1: MAE = 265.46 MW
Fold 2: MAE = 260.21 MW
Fold 3: MAE = 220.82 MW

Average CV MAE: 248.83 MW
Best fold MAE: 220.82 MW


In [13]:
# Cell 0: Setup paths and create folders
import os
from pathlib import Path

# Create necessary folders
Path('models').mkdir(parents=True, exist_ok=True)
Path('reports').mkdir(parents=True, exist_ok=True)

print("✅ Folders created: models/, reports/")
print(f"Current directory: {os.getcwd()}")

✅ Folders created: models/, reports/
Current directory: d:\sri-lanka-renewable-forecast


In [14]:
# Cell 8: Save the Best Model
import pickle

# Save to local 'models' folder (not ../models)
with open('models/renewable_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("✅ Model saved to: models/renewable_model.pkl")

✅ Model saved to: models/renewable_model.pkl


In [ ]:
# Feature Importance (Only for Tree-based Models)
if best_model_name in ['LightGBM', 'XGBoost', 'Random Forest']:
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Feature Importance:")
    print(importance.head(10))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.barh(importance['feature'][:10], importance['importance'][:10], color='#008080')
    plt.xlabel('Importance')
    plt.title(f'Top 10 Features - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    #plt.savefig('../reports/feature_importance.png', dpi=150)
    plt.show()
else:
    print(f"Feature importance not available for {best_model_name} (linear model)")

NameError: name 'best_model_name' is not defined

In [16]:
#Final Evaluation on 2024 Holdout
print("=" * 60)
print("FINAL EVALUATION ON 2024 DATA")
print("=" * 60)

# Split by year
train = df[df.index.year < 2024]
test = df[df.index.year == 2024]

X_train = train[feature_cols].ffill().fillna(0)
y_train = train['renewable_target'].dropna()
X_test = test[feature_cols].ffill().fillna(0)
y_test = test['renewable_target'].dropna()

# Align indices
X_train = X_train.loc[y_train.index]
X_test = X_test.loc[y_test.index]

# Train final model on all pre-2024 data
if best_model_name == 'LightGBM':
    final_model = lgb.LGBMRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=-1)
elif best_model_name == 'XGBoost':
    final_model = xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.05, random_state=42, verbosity=0)
elif best_model_name == 'Random Forest':
    final_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
elif best_model_name == 'Ridge':
    final_model = Ridge(alpha=1.0)
else:
    final_model = LinearRegression()

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"\nTest on 2024 data:")
print(f"  Samples: {len(y_test)}")
print(f"  Mean actual: {y_test.mean():.1f} MW")
print(f"  MAE: {mae:.2f} MW")
print(f"  RMSE: {rmse:.2f} MW")
print(f"  MAPE: {mape:.1f}%")
print(f"  Relative error: {mae / y_test.mean() * 100:.1f}%")

FINAL EVALUATION ON 2024 DATA

Test on 2024 data:
  Samples: 35133
  Mean actual: 719.3 MW
  MAE: 222.90 MW
  RMSE: 279.71 MW
  MAPE: 31.8%
  Relative error: 31.0%


In [17]:
# Calculate Prediction Intervals
errors = y_test - y_pred
lower_bound = np.percentile(errors, 10)
upper_bound = np.percentile(errors, 90)

print("=" * 60)
print("PREDICTION INTERVALS")
print("=" * 60)
print(f"Error distribution on 2024 test set:")
print(f"  10th percentile (lower): {lower_bound:.1f} MW")
print(f"  50th percentile (median): {np.percentile(errors, 50):.1f} MW")
print(f"  90th percentile (upper): {upper_bound:.1f} MW")

print(f"\nFor a new prediction:")
print(f"  80% confidence interval = [prediction + {lower_bound:.0f}, prediction + {upper_bound:.0f}]")
print(f"  Example: If prediction = 650 MW, then 80% interval = {650 + lower_bound:.0f} to {650 + upper_bound:.0f} MW")

PREDICTION INTERVALS
Error distribution on 2024 test set:
  10th percentile (lower): -230.8 MW
  50th percentile (median): 111.9 MW
  90th percentile (upper): 447.6 MW

For a new prediction:
  80% confidence interval = [prediction + -231, prediction + 448]
  Example: If prediction = 650 MW, then 80% interval = 419 to 1098 MW


In [18]:
# Sample Prediction Function
def predict_renewable(temp, solar, wind, rain, humidity, hour=12, day_of_week=2, month=4):
    """
    Predict renewable generation from weather inputs.
    
    Parameters:
    - temp: temperature (°C)
    - solar: solar radiation (W/m²)
    - wind: wind speed (m/s)
    - rain: rainfall (mm)
    - humidity: humidity (%)
    - hour: hour of day (0-23)
    - day_of_week: Monday=0, Sunday=6
    - month: 1-12
    """
    # Create time features
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    dow_sin = np.sin(2 * np.pi * day_of_week / 7)
    dow_cos = np.cos(2 * np.pi * day_of_week / 7)
    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    is_weekend = 1 if day_of_week >= 5 else 0
    
    # Create feature array
    features = np.array([[
        solar, temp, wind, rain, humidity,
        hour_sin, hour_cos, dow_sin, dow_cos, month_sin, month_cos, is_weekend
    ]])
    
    # Predict
    prediction = final_model.predict(features)[0]
    
    # Add uncertainty from test set errors
    lower = prediction + lower_bound
    upper = prediction + upper_bound
    
    return {
        'prediction_mw': prediction,
        'lower_80_mw': lower,
        'upper_80_mw': upper,
        'message': f"We are 80% confident renewable generation will be between {lower:.0f} and {upper:.0f} MW"
    }

# Test the function
print("\n" + "=" * 60)
print("SAMPLE PREDICTION")
print("=" * 60)
test_pred = predict_renewable(temp=32, solar=500, wind=4, rain=0, humidity=65, hour=12, day_of_week=2, month=4)
print(f"Input: 32°C, 500 W/m² solar, 4 m/s wind, 0 mm rain, 65% humidity")
print(f"\nBest estimate: {test_pred['prediction_mw']:.0f} MW")
print(f"80% interval: {test_pred['lower_80_mw']:.0f} - {test_pred['upper_80_mw']:.0f} MW")
print(f"\n{test_pred['message']}")


SAMPLE PREDICTION
Input: 32°C, 500 W/m² solar, 4 m/s wind, 0 mm rain, 65% humidity

Best estimate: 294 MW
80% interval: 63 - 741 MW

We are 80% confident renewable generation will be between 63 and 741 MW


In [20]:
# Summary Report
print("\n" + "=" * 60)
print("FINAL SUMMARY - RENEWABLE FORECAST MODEL")
print("=" * 60)

summary = {
    'Best Model': best_model_name,
    'Cross-Validation MAE (MW)': f"{np.mean(cv_results):.2f} ± {np.std(cv_results):.2f}",
    'Test MAE (2024) - MW': f"{mae:.2f}",
    'Test RMSE (2024) - MW': f"{rmse:.2f}",
    'Test MAPE (2024) - %': f"{mape:.1f}",
    'Prediction Coverage': "80% (10th-90th percentile)",
    'Lower Error Bound (MW)': f"{lower_bound:.1f}",
    'Upper Error Bound (MW)': f"{upper_bound:.1f}",
    'Features Used': len(feature_cols),
    'Training Period': "2021-2023",
    'Test Period': "2024"
}

for key, value in summary.items():
    print(f"{key}: {value}")

print("\n" + "=" * 60)
print("✅ RENEWABLE FORECAST MODEL READY")
print("=" * 60)


FINAL SUMMARY - RENEWABLE FORECAST MODEL
Best Model: LightGBM
Cross-Validation MAE (MW): 248.83 ± 19.93
Test MAE (2024) - MW: 222.90
Test RMSE (2024) - MW: 279.71
Test MAPE (2024) - %: 31.8
Prediction Coverage: 80% (10th-90th percentile)
Lower Error Bound (MW): -230.8
Upper Error Bound (MW): 447.6
Features Used: 12
Training Period: 2021-2023
Test Period: 2024

✅ RENEWABLE FORECAST MODEL READY


In [21]:
from pathlib import Path

# Check current directory
print(f"Current directory: {Path.cwd()}")

# Look for model file
model_path = Path('models/renewable_model.pkl')
print(f"Model exists: {model_path.exists()}")

if model_path.exists():
    print(f"Model size: {model_path.stat().st_size / 1024:.1f} KB")
else:
    # Try alternative paths
    for p in Path('.').rglob('renewable_model.pkl'):
        print(f"Found model at: {p}")

Current directory: d:\sri-lanka-renewable-forecast
Model exists: True
Model size: 570.8 KB
